# Umid LLM — hammasi bitta katakda (Colab / Kaggle, GPU)

Bu katak quyidagilarni ketma-ket bajaradi:
1. **Ma'lumot.** O'zbekcha Vikipediya va yangiliklar yuklanadi va tozalanadi. Tozalash: belgilar, URL, vikimatn,
   "Manbalar" bo'limlari, kirillcha va takror hujjatlar.
2. **Tokenizer va tokenlar.**
3. **Pretraining.** Algoritm — Tanlov xotirasi. Model eng muhim tokenlarni o'zi tanlab saqlaydi, xotira o'zgarmas.
4. **Sekin SFT.** O'rganish tezligi kichik, qadamlar ko'p. Ma'lumot: loyiha savol-javoblari va o'zbekcha ko'rsatmalar.
5. **Baholash** (60 ta yangi savol) va namunaviy javoblar.

**Boshlashdan oldin:**
- Colab: *Runtime → Change runtime type → T4 GPU*.
- Kaggle: *Settings → Accelerator → GPU*, *Internet → On*.

**Sessiya uzilsa**, xuddi shu katakni qayta ishga tushiring. Tugagan bosqichlar o'tkazib yuboriladi,
pretraining to'xtagan joyidan davom etadi. Natijalar Google Drive'ga (Colab) yoki `/kaggle/working` ga saqlanadi.

**Tez sinov uchun** avval `MAX_MB = 50`, `STEPS = 1000`, `SFT_STEPS = 300` qilib ko'ring.

In [ ]:
# ===== Sozlamalar =====
MIXER     = "tanlov"   # "tanlov" — yangi algoritm; "delta" — eski Umid
PRESET    = "small"    # tiny (~7.6M) | small (~17.7M) | base (~104M parametr)
MAX_MB    = 600        # qancha matn (MB); tez sinov: 50
STEPS     = 20000      # pretraining qadamlari; tez sinov: 1000
SFT_STEPS = 6000       # sekin SFT qadamlari; tez sinov: 300
SFT_LR    = 1e-4       # SFT o'rganish tezligi (kichik — pretraining bilimini buzmaydi)
COMPARE_GPT = False    # True — xuddi shu sharoitda Transformer ham o'qitilib, solishtiriladi
# =======================

import os
try:                                   # Colab: natijalar Google Drive'ga
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE = "/content/drive/MyDrive/umid"
except ImportError:                    # Kaggle yoki boshqa joy
    SAVE = "/kaggle/working/umid" if os.path.exists("/kaggle/working") else os.path.abspath("umid_natijalar")

if not os.path.exists("pipeline.py"):
    if not os.path.exists("GPT_UZ"):
        !git clone -q https://github.com/ABDULHODIY1/GPT_UZ.git
    %cd GPT_UZ
!git pull -q
!pip install -q datasets
!nvidia-smi --query-gpu=name,memory.total --format=csv

extra = "--compare-gpt" if COMPARE_GPT else ""
!python pipeline.py --save {SAVE} --mixer {MIXER} --preset {PRESET} --max-mb {MAX_MB} --steps {STEPS} --sft-steps {SFT_STEPS} --sft-lr {SFT_LR} {extra}

### (Ixtiyoriy) Suhbat

Colab'da interaktiv ishlaydi. Kaggle'da kiritish oynasi yo'q, u yerda yuqoridagi katak chiqargan namunaviy javoblarni ko'ring.

In [ ]:
!python chat.py --ckpt {SAVE}/chat/model.pt --device cuda